In [1]:
from numba import njit, prange
import numpy as np

import faiss

data = np.load("/home/dhem/workspace/2024.3/data/save/train_test-final-62512502500.npz")

x = data["x"]
y = data["y"]
w = data["w"]
coor = data["coor"]
name = data["name"]

In [2]:
(num_sample, dim_sample) = x.shape
res = faiss.StandardGpuResources()
flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
index = faiss.GpuIndexFlatL2(res, dim_sample, flat_config)
index.add(x)

In [3]:
b = x.copy()
min_number = 4
distances, indices = index.search(b, min_number)

var_y = np.var(np.einsum("ij,i->ij", y[indices], x[:, 0] * w), axis=1)
argsort_ = np.argsort(var_y)[::-1][:100]
print(np.einsum("ij,i->ij", y[indices], x[:, 0] * w)[argsort_])

[[-2.45426138e-05 -2.14069595e-05 -2.27201855e-05  1.91334897e-06]
 [-1.92273005e-05 -1.67247609e-05 -2.13789361e-05  5.18815543e-06]
 [-2.19398581e-05 -2.26885914e-05 -1.64835116e-05  1.50115022e-07]
 [-2.02617480e-05 -1.55383507e-05 -1.05284021e-06 -1.02114601e-06]
 [-2.18427090e-05 -2.25506768e-05 -1.65314870e-05 -1.38046941e-06]
 [-2.45493762e-05 -2.46365530e-05 -2.47684752e-05 -5.29377696e-06]
 [-1.50765251e-05 -1.60172315e-05  1.15793806e-06  1.12526822e-06]
 [-2.83364100e-05 -2.54586224e-05 -1.04651827e-05 -1.04453495e-05]
 [-3.46810223e-05 -2.03040723e-05 -2.01730150e-05 -3.83418730e-05]
 [-2.49527222e-05 -2.43002180e-05 -2.46706688e-05 -5.63676285e-06]
 [-3.50213165e-05 -3.45228190e-05 -3.78337159e-05 -1.71131128e-05]
 [-3.44057697e-05 -3.49025770e-05 -3.77054410e-05 -1.70550909e-05]
 [-3.57163715e-05 -3.41986036e-05 -1.87860852e-05 -1.87037408e-05]
 [-1.40446862e-05 -9.24621897e-06  4.23044529e-06  4.24941018e-06]
 [-2.46408531e-05 -2.39066457e-05 -2.33955122e-05 -5.38530736e

In [8]:
distance = np.sum(
    np.transpose(
        np.transpose(x[indices[argsort_]], axes=(0, 2, 1)) - x[argsort_][:, :, None],
        axes=(0, 2, 1),
    ) ** 2,
    axis=2,
)
# print(x[argsort_][:, :, None].shape)
# print(np.transpose(x[indices[argsort_]], axes=(0, 2, 1)).shape)
energy = np.einsum("ij,i->ij", y[indices[argsort_]], (x[:, 0] * w)[argsort_])
print(y[indices[argsort_]] - y[argsort_][:, None])
print(energy)

[[ 0.00000000e+00  2.13207422e-03  1.23915203e-03  1.79886147e-02]
 [ 0.00000000e+00  1.89175247e-03 -1.62649259e-03  1.84564513e-02]
 [ 0.00000000e+00 -5.57398514e-04  4.06200655e-03  1.64449995e-02]
 [ 0.00000000e+00  3.74029397e-03  1.52108656e-02  1.52359631e-02]
 [ 0.00000000e+00 -5.23035859e-04  3.92385005e-03  1.51171917e-02]
 [ 0.00000000e+00 -4.98369674e-05 -1.25253874e-04  1.10079839e-02]
 [ 0.00000000e+00 -9.93460274e-04  1.71448754e-02  1.71103734e-02]
 [ 0.00000000e+00  1.77062791e-03  1.09957016e-02  1.10079044e-02]
 [ 0.00000000e+00  6.93343764e-03  6.99664142e-03 -1.76548433e-03]
 [ 0.00000000e+00  3.70187478e-04  1.60018332e-04  1.09585906e-02]
 [ 0.00000000e+00  2.39557903e-04 -1.35152641e-03  8.60596478e-03]
 [ 0.00000000e+00 -2.39557903e-04 -1.59108431e-03  8.36640688e-03]
 [ 0.00000000e+00  6.68415195e-04  7.45598916e-03  7.49225307e-03]
 [ 0.00000000e+00  4.34202342e-03  1.65367492e-02  1.65539101e-02]
 [ 0.00000000e+00  4.22511469e-04  7.16651435e-04  1.10809138e

In [5]:
from matplotlib import pyplot as plt

color_dict = {
    "methane_cc-pVDZ_0-1_1_-0.2000": "#004D40",
    "methane_cc-pVDZ_0-1_1_-0.1000": "#1A237E",
    "methane_cc-pVDZ_0-1_1_0.0000": "#7B1FA2",
    "methane_cc-pVDZ_0-1_1_0.1000": "#B71C1C",
    "methane_cc-pVDZ_0-1_1_0.2000": "#FF6F00",
}

plt.rcParams["figure.figsize"] = np.array([3, 3]) * 520 / 72

f, axes = plt.subplots(10, 10)
axes = axes.reshape(10, 10)

begin_y = 0.025
end_y = 0.95
int_y = 0.0
begin_x = 0.025
end_x = 0.95
int_x = 0.0
end_x += int_x
end_y += int_y

shapexy = np.shape(axes)
inter_x = np.linspace(begin_x, end_x, shapexy[1] + 1)
inter_y = np.linspace(begin_y, end_y, shapexy[0] + 1)

delta_x = inter_x[1] - inter_x[0] - int_x
delta_y = inter_y[1] - inter_y[0] - int_y

for i in range(shapexy[0]):
    for j in range(shapexy[1]):
        axes[i][j].set_position(
            [
                inter_x[j],
                inter_y[i],
                inter_x[j + 1] - inter_x[j] - int_x,
                inter_y[i + 1] - inter_y[i] - int_y,
            ]
        )
        axes[i][j].xaxis.set_tick_params(
            direction="in", which="both", bottom=True, top=True
        )
        axes[i][j].yaxis.set_tick_params(
            direction="in", which="both", left=True, right=True
        )
        if i != 0:
            axes[i][j].set_xticks([])
        if j != 0:
            axes[i][j].set_yticks([])


for i in range(argsort_.shape[0]):
    axes_i, axes_j = np.unravel_index(i, (10, 10))
    for j in range(indices.shape[1]):
        axes[axes_i, axes_j].scatter(
            distance[i][j],
            np.abs(energy[i][j] - energy[i][0]) * 627.509,
            c=color_dict[name[indices[argsort_[i]]][j]],
        )
        axes[axes_i, axes_j].set_xlim(-0.002, 0.022)
        axes[axes_i, axes_j].set_ylim(-1e-3, 1.1e-2)
plt.savefig("test.pdf", dpi=300)
plt.clf()

ModuleNotFoundError: No module named 'matplotlib'